In [1]:
import h5py
from pathlib import Path
import sys

In [2]:
sys.argv

['/Users/vsmw51/opt/anaconda3/envs/py313/lib/python3.13/site-packages/ipykernel_launcher.py',
 '--f=/Users/vsmw51/Library/Jupyter/runtime/kernel-v3765c96ba73aaa93c7ef7cc5b5f975ee694aaf936.json']

In [4]:
len(sys.argv)

2

In [2]:
fn = Path("/opt/topspin4.5.0/examdata/exam_CMCse_1/alpha_ionone.jjh5")

In [3]:
with h5py.File(fn, "r") as f:
    print("Keys:", list(f['JasonDocument']))  
    print("Keys: Molecules\n\t", list(f['JasonDocument']['Molecules'])) 
    print("Keys: Items\n\t", list(f['JasonDocument']['Items']))   
    print("Keys: NMR\n\t", list(f['JasonDocument']['NMR']))   
    # Explore the structure of the HDF5 file

Keys: ['Items', 'Molecules', 'NMR']
Keys: Molecules
	 ['Molecules', 'SpecData']
Keys: Items
	 ['0', '1', '2', '3', '4', '5', '6']
Keys: NMR
	 ['NMRData']


In [11]:
with h5py.File(fn, "r") as f:
    # print("Keys:", list(f['JasonDocument']))  
    # print("Keys: NMR / NMRData\n\t", list(f['JasonDocument']['NMR']['NMRData']))
    # print("Keys: NMR / NMRData / 0\n\t", list(f['JasonDocument']['NMR']['NMRData']['0']))
    # # print("Keys: NMR / NMRData / 0 / SpecInfo / 0\n\t", list(f['JasonDocument']['NMR']['NMRData']['0']['SpecInfo'].attrs['SpectrumType']))
    print("Keys: NMR / NMRData / 0 / Multiplets_Integrals / 0\n\t", list(f['JasonDocument']['NMR']['NMRData']['2']['Multiplets_Integrals']))


Keys: NMR / NMRData / 0 / Multiplets_Integrals / 0
	 []


In [5]:
with h5py.File(fn, 'r') as f:
    try:
        peakList = f['JasonDocument']['NMR']['NMRData']['3']['Peaks']['PeakList']
        for pk in peakList:
            attributes = peakList[pk].attrs
            print(f"@{pk}: {attributes['Pos']}")
    except KeyError as e:
        print(f"Path not found: {e}")

@0: [ 1.43858064 22.96129083  0.        ]
@1: [ 1.20189873 22.96129083  0.        ]
@2: [ 0.90768475 27.66011516  0.        ]
@3: [ 0.90622668 31.1195029   0.        ]
@4: [ 0.90622609 54.25692566  0.        ]
@5: [ 0.90549764 32.52606715  0.        ]
@6: [ 0.83332297 31.1955334   0.        ]
@7: [ 0.8318649 32.4880519  0.       ]
@8: [ 0.83089227 54.27921046  0.        ]


In [6]:

def find_datasets_with_peaks(filename):
    """
    Loop through NMRData datasets and find which ones contain peaks.
    
    Args:
        filename (str): Path to the HDF5 file
        
    Returns:
        dict: Dictionary with dataset names as keys and peak information as values
    """
    datasets_with_peaks = {}
    
    with h5py.File(filename, 'r') as f:
        try:
            nmr_data = f['JasonDocument']['NMR']['NMRData']
            
            # Loop through all datasets in NMRData
            for dataset_name in nmr_data.keys():
                print(f"Checking dataset: {dataset_name}")
                
                try:
                    dataset = nmr_data[dataset_name]
                    
                    # Check if Peaks group exists
                    if 'Peaks' in dataset:
                        peaks_group = dataset['Peaks']
                        peak_info = {
                            'has_peaks_group': True,
                            'has_peaklist': False,
                            'peak_count': 0,
                            'available_keys': list(peaks_group.keys())
                        }
                        
                        # Check if PeakList exists
                        if 'PeakList' in peaks_group:
                            peaklist = peaks_group['PeakList']
                            peak_info['has_peaklist'] = True
                            peak_info['peak_count'] = len(peaklist.keys())
                            peak_info['peak_names'] = list(peaklist.keys())
                            
                        datasets_with_peaks[dataset_name] = peak_info
                        
                    else:
                        # No Peaks group found
                        datasets_with_peaks[dataset_name] = {
                            'has_peaks_group': False,
                            'available_keys': list(dataset.keys()) if hasattr(dataset, 'keys') else []
                        }
                        
                except Exception as e:
                    print(f"  Error accessing dataset {dataset_name}: {e}")
                    datasets_with_peaks[dataset_name] = {'error': str(e)}
                    
        except KeyError as e:
            print(f"Could not find NMRData path: {e}")
            return None
            
    return datasets_with_peaks

def print_peak_summary(filename):
    """
    Print a summary of datasets with peaks.
    """
    results = find_datasets_with_peaks(filename)
    
    if results is None:
        print("Could not access NMRData")
        return
    
    print("\n" + "="*50)
    print("PEAK SUMMARY")
    print("="*50)
    
    datasets_with_peaklists = []
    datasets_with_peaks_only = []
    datasets_without_peaks = []
    
    for dataset_name, info in results.items():
        if 'error' in info:
            print(f"❌ {dataset_name}: Error - {info['error']}")
        elif info.get('has_peaklist', False):
            datasets_with_peaklists.append((dataset_name, info['peak_count']))
            print(f"✅ {dataset_name}: Has PeakList with {info['peak_count']} peaks")
        elif info.get('has_peaks_group', False):
            datasets_with_peaks_only.append(dataset_name)
            print(f"⚠️  {dataset_name}: Has Peaks group but no PeakList")
            print(f"   Available: {info['available_keys']}")
        else:
            datasets_without_peaks.append(dataset_name)
            print(f"❌ {dataset_name}: No Peaks group")
    
    print(f"\nSummary:")
    print(f"  Datasets with PeakList: {len(datasets_with_peaklists)}")
    print(f"  Datasets with Peaks group only: {len(datasets_with_peaks_only)}")
    print(f"  Datasets without peaks: {len(datasets_without_peaks)}")
    
    return results

# Usage example:
def extract_all_peaks(filename):
    """
    Extract ppm values from all datasets that have PeakList.
    """
    results = find_datasets_with_peaks(filename)
    
    if results is None:
        return
    
    with h5py.File(filename, 'r') as f:
        for dataset_name, info in results.items():
            if info.get('has_peaklist', False):
                print(f"\n--- Peaks from dataset {dataset_name} ---")
                try:
                    peakList = f['JasonDocument']['NMR']['NMRData'][dataset_name]['Peaks']['PeakList']
                    for pk in peakList:
                        attributes = peakList[pk].attrs
                        print(f"@{pk}: {attributes['Pos']} ppm")
                except Exception as e:
                    print(f"Error extracting peaks: {e}")


In [12]:
find_datasets_with_peaks(fn)

Checking dataset: 0
Checking dataset: 1
Checking dataset: 2
Checking dataset: 3
Checking dataset: 4
Checking dataset: 5


{'0': {'has_peaks_group': True,
  'has_peaklist': False,
  'peak_count': 0,
  'available_keys': []},
 '1': {'has_peaks_group': True,
  'has_peaklist': True,
  'peak_count': 11,
  'available_keys': ['AutoPPOptions', 'PeakList'],
  'peak_names': ['0', '1', '10', '2', '3', '4', '5', '6', '7', '8', '9']},
 '2': {'has_peaks_group': True,
  'has_peaklist': False,
  'peak_count': 0,
  'available_keys': []},
 '3': {'has_peaks_group': True,
  'has_peaklist': True,
  'peak_count': 9,
  'available_keys': ['AutoPPOptions', 'PeakList'],
  'peak_names': ['0', '1', '2', '3', '4', '5', '6', '7', '8']},
 '4': {'has_peaks_group': True,
  'has_peaklist': True,
  'peak_count': 12,
  'available_keys': ['AutoPPOptions', 'PeakList'],
  'peak_names': ['0',
   '1',
   '10',
   '11',
   '2',
   '3',
   '4',
   '5',
   '6',
   '7',
   '8',
   '9']},
 '5': {'has_peaks_group': True,
  'has_peaklist': False,
  'peak_count': 0,
  'available_keys': []}}

In [7]:

filename = fn # Replace with your filename

# Get detailed summary
print_peak_summary(filename)

# Extract all peaks
extract_all_peaks(filename)

Checking dataset: 0
Checking dataset: 1
Checking dataset: 2
Checking dataset: 3
Checking dataset: 4
Checking dataset: 5

PEAK SUMMARY
⚠️  0: Has Peaks group but no PeakList
   Available: []
✅ 1: Has PeakList with 11 peaks
⚠️  2: Has Peaks group but no PeakList
   Available: []
✅ 3: Has PeakList with 9 peaks
✅ 4: Has PeakList with 12 peaks
⚠️  5: Has Peaks group but no PeakList
   Available: []

Summary:
  Datasets with PeakList: 3
  Datasets with Peaks group only: 3
  Datasets without peaks: 0
Checking dataset: 0
Checking dataset: 1
Checking dataset: 2
Checking dataset: 3
Checking dataset: 4
Checking dataset: 5

--- Peaks from dataset 1 ---
@0: [  6.59263    148.97472147   0.        ] ppm
@1: [  6.02824836 132.25041524   0.        ] ppm
@10: [ 0.83035389 27.81632065  0.        ] ppm
@2: [  5.47703576 122.57274394   0.        ] ppm
@3: [ 2.26069841 54.25952748  0.        ] ppm
@4: [ 2.23090547 26.9258866   0.        ] ppm
@5: [ 2.0267751 23.0350259  0.       ] ppm
@6: [ 1.54506941 22.57

In [73]:
with h5py.File(fn, 'r') as f:
    peaks = f['JasonDocument']['NMR']['NMRData']['3']['Peaks']
    peakList = peaks['PeakList']
    peak0 = peakList['8']
    attributes = peak0.attrs
    print("Keys: peaks\n\t", list(peaks))
    print("Keys: peakList\n\t", list(peakList))
    print("Keys: peak0\n\t", list(peak0))

    for k, v in attributes.items():
        print(f"@{k}: {v}")

Keys: peaks
	 ['AutoPPOptions', 'PeakList']
Keys: peakList
	 ['0', '1', '2', '3', '4', '5', '6', '7', '8']
Keys: peak0
	 []
@CovMat: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0.]
@Height: 1403897160.5772004
@ID: b'{e426a912-f6f5-467f-9dd8-6f5ccdb9716b}'
@Offset: 0.0
@PeakClassification: 0
@PeakLabel: b''
@PeakScope: 2
@PeakType: 0
@Pos: [ 0.83089227 54.27921046  0.        ]
@ShapePar: [0. 0. 0.]
@Width: [0.00215157 0.21547118 0.        ]
@fFitParamsHaveBeenManuallyEdited: 0


In [63]:
with h5py.File(fn, 'r') as f:
    attributes = f['JasonDocument']['NMR']['NMRData']['4']['SpecInfo'].attrs
    for k, v in attributes.items():
        print(f"@{k}: {v}")


@Arrayed: 0
@AutoPeakType: 0
@AutoProcessingMode: 0
@AutoProcessingMode.str: b'rawdata'
@ExperimentType: 1
@ExperimentType.str: b'Conventional pulse acquire'
@OrigFileFormat: 6
@OrigFileFormat.str: b'TopSpin'
@OrigFilename: b'/opt/topspin4.5.0/examdata/exam_CMCse_1/5/fid'
@OrigFilename.filename.str: b'fid'
@OrigFilename.filepath.str: b'/opt/topspin4.5.0/examdata/exam_CMCse_1/5'
@PulseProgram: b'zgpg30'
@SW: [2.40384615e+04 1.00000000e+00 1.00000000e+00 1.00000000e+01
 1.00000000e+01 1.00000000e+01 1.00000000e+01 1.00000000e+01]
@Solvent: b'CDCl3'
@SpectrometerFrequencies: [100.6228298   0.          0.          0.          0.          0.
   0.          0.       ]
@SpectrumRef: [10060.802853     0.           0.           0.           0.
     0.           0.           0.      ]
@SpectrumType: [1 0 0 0 0 0 0 0]
@SpectrumTypeExt: [ 1 -1 -1 -1 -1 -1 -1 -1]
@SpectrumTypeExt.strs: ['Frequency domain, spectrum']
@Spinrate: 4200.0
@Temperature: 26.55000000000001
@Title: b'exam_CMCse_1'
@appliedP

In [ ]:
with h5py.File(fn, "r") as f:
    print("Keys:", list(f['JasonDocument']))  
    print("Keys: Molecules\n\t", list(f['JasonDocument']['Molecules'])) 
    print("Keys: Molecules / Molecules\n\t", list(f['JasonDocument']['Molecules']['Molecules'])) 
    print("Keys: Molecules / Molecules / 0\n\t", list(f['JasonDocument']['Molecules']['Molecules']['0'])) 
    print("Keys: Molecules / Molecules / 0 / Atoms\n\t", list(f['JasonDocument']['Molecules']['Molecules']['0']['Atoms'])) 
    print("Keys: Molecules / Molecules / 0 / Atoms / 0\n\t", list(f['JasonDocument']['Molecules']['Molecules']['0']['Atoms']['0'].attrs)) 
    print("Keys: Molecules / Molecules / 0 / Atoms / 0\n\t", list(f['JasonDocument']['Molecules']['Molecules']['0']['Atoms']['0'].attrs)) 


    for k, v in f['JasonDocument']['Molecules']['Molecules']['0']['NMRData'].attrs.items():
        print(f"@{k}: {v}")


Keys: ['Items', 'Molecules', 'NMR']
Keys: Molecules
	 ['Molecules', 'SpecData']
Keys: Molecules / Molecules
	 ['0']
Keys: Molecules / Molecules / 0
	 ['Atoms', 'NMRData', 'Rings', 'Symmetry']
Keys: Molecules / Molecules / 0 / Atoms
	 ['0', '1', '10', '11', '12', '13', '2', '3', '4', '5', '6', '7', '8', '9']
Keys: Molecules / Molecules / 0 / Atoms / 0
	 ['El', 'NB.Conn', 'NB.Num', 'X', 'Y', 'nH']
Keys: Molecules / Molecules / 0 / Atoms / 0
	 ['El', 'NB.Conn', 'NB.Num', 'X', 'Y', 'nH']


In [56]:
with h5py.File(fn, 'r') as f:
    print(list(f['JasonDocument']['Molecules']['Molecules']['0']['NMRData']))
    print(list(f['JasonDocument']['Molecules']['Molecules']['0']['NMRData']['Spectra']['SpectraList']['0']['Shifts']['0'].attrs))
    attributes = f['JasonDocument']['Molecules']['Molecules']['0']['NMRData']['Spectra']['SpectraList']['0']['Shifts']['0'].attrs
    for k, v in attributes.items():
       print(f"@{k}: {v}")


['Couplings', 'Spectra']
['ACount', 'ErrorSpheres', 'IgnoredAuto', 'IgnoredUser', 'IsExchangeable', 'Nums', 'Value', 'Value.Error', 'Value.Method', 'ValueSpheres', 'nH']
@ACount: 1
@ErrorSpheres: 0
@IgnoredAuto: 0
@IgnoredUser: 0
@IsExchangeable: 0
@Nums: [0]
@Value: [198.01193 197.2     197.2    ]
@Value.Error: [-1.         5.6267786  5.6267786]
@Value.Method: [2 4 5]
@ValueSpheres: 6
@nH: 0


In [26]:
with h5py.File(fn, "r") as f:
    print(f['JasonDocument']['Molecules'].attrs.keys())

<KeysViewHDF5 []>


In [ ]:
def print_structure(name, obj):
    print(name)
    if hasattr(obj, 'attrs'):
        for key, val in obj.attrs.items():
            print(f"  @{key}: {val}")

with h5py.File(fn, 'r') as f:
    f.visititems(print_structure)

JasonDocument
  @AllLogFilePaths: ['']
  @DPI: 72.0
  @HPages: 3
  @LogFilePath: b'/Users/vsmw51/.jason/logs/2025-08-14T06-53-37.309_53319.log'
  @Margins: [0. 0. 0. 0.]
  @Orientation: 1
  @PageSize: [595 842]
  @Units: 1
  @VPages: 2
JasonDocument/Items
  @.container_type: 9
JasonDocument/Items/0
  @FeaturesFlags: 0
  @Geometry: [ 10.  10. 822. 575.]
  @ID: b'{5bbfc00a-6269-4f01-b882-586a66c71233}'
  @ItemVersion: 1
  @Pos: [10. 10.]
  @Rotation: 0.0
  @ScenePos: [10. 10.]
  @TransformOrigPoint: [0. 0.]
  @Type: 65538
  @ZValue: 0.0
JasonDocument/Items/0/BackGroundBrush
  @Color: [  0   0   0 255   1]
  @Style: 0
JasonDocument/Items/0/BorderPen
  @Color: [204 204 204 255   1]
  @Style: 1
  @Width: 1
JasonDocument/Items/0/NMRPlot
  @ActiveBold: 0
  @Antialiasing: 1
  @Ceiling: 100.0
  @ClipInStack: 0
  @ColourCont: 0
  @DeclutterPeakLabels: 1
  @DiagonalSlope: -1.0
  @Floor: -1.0
  @Header: b'/opt/topspin4.5.0/examdata/exam_CMCse_1/1/pdata/1/1r<br>Alpha Ionone\nC13H20O in CDCl3'
  @He